In [1]:
#Environment Setup & Installations
import os
import sys
import subprocess
import glob
import shutil
import nest_asyncio

# 1. Set environment variables for the model cache
os.environ.setdefault("HF_HOME", "/kaggle/working/hf-cache")
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
nest_asyncio.apply()

print("📦 Installing core dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", 
                "vllm", "mistral-common>=1.8.6", "openai", "huggingface_hub", 
                "easyocr", "scikit-image", "opencv-python-headless", 
                "ipywidgets", "nest_asyncio", "bitsandbytes", "transformers<5.17.0"], check=False)

print("🔧 Cleaning up broken Pillow installations...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "Pillow", "PIL", "pillow"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
site_packages = "/usr/local/lib/python3.12/dist-packages"
for pattern in ["PIL", "Pillow-*.dist-info", "Pillow-*.egg-info", "PIL-*.dist-info", "PIL-*.egg-info"]:
    for path in glob.glob(os.path.join(site_packages, pattern)):
        shutil.rmtree(path, ignore_errors=True)

print("📦 Reinstalling clean Pillow 12.0.0...")
subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "--no-deps", "Pillow==12.0.0"], check=True)

# 2. Setup Configuration & Folders
KAGGLE_BASE = "/kaggle/working/ui_converter"
CONFIG = {
    "output_path": f"{KAGGLE_BASE}/output/",
    "input_path": f"{KAGGLE_BASE}/input/",
    "ai_engine": "devstral_24b",
}
os.makedirs(CONFIG["output_path"], exist_ok=True)
os.makedirs(CONFIG["input_path"], exist_ok=True)

import torch
print("\n✅ Environment Setup Complete!")
print(f"🖥️ GPUs Available: {torch.cuda.device_count()} ({torch.cuda.get_device_name(0)})")

📦 Installing core dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 58.2 MB/s eta 0:00:00
🔧 Cleaning up broken Pillow installations...
📦 Reinstalling clean Pillow 12.0.0...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 93.7 MB/s eta 0:00:00

✅ Environment Setup Complete!
🖥️ GPUs Available: 2 (Tesla T4)


In [2]:
#Image Processing Utilities

import os
import cv2
import numpy as np
from typing import Dict, Any

class ImageProcessor:
    @staticmethod
    def preprocess(image_path: str, output_dir: str) -> Dict[str, Any]:
        filename = os.path.basename(image_path)
        img = cv2.imread(image_path)
        if img is None:
            raise ValueError(f"Invalid image file: {image_path}")

        height, width = img.shape[:2]
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced_gray = clahe.apply(gray)
        edges = cv2.Canny(enhanced_gray, 50, 150)

        proc_dir = os.path.join(output_dir, "processed")
        os.makedirs(proc_dir, exist_ok=True)
        proc_path = os.path.join(proc_dir, f"enhanced_{filename}")
        cv2.imwrite(proc_path, enhanced_gray)

        return {
            "original_path": image_path,
            "processed_path": proc_path,
            "edges": edges,
            "image": img,
            "dimensions": {"width": width, "height": height}
        }

print("✅ Image Utilities Loaded")

✅ Image Utilities Loaded


In [3]:
#The vLLM AI Server
import os
import time
import json
import base64
import mimetypes
import shutil
import subprocess
import urllib.request
from pathlib import Path
import torch
from openai import OpenAI

MODEL_ID = "mistralai/Devstral-Small-2-24B-Instruct-2512"
DEVSTRAL_HOST = "127.0.0.1"
DEVSTRAL_PORT = 8000
DEVSTRAL_BASE_URL = f"http://{DEVSTRAL_HOST}:{DEVSTRAL_PORT}/v1"
DEVSTRAL_HEALTH_URL = f"http://{DEVSTRAL_HOST}:{DEVSTRAL_PORT}/v1/models"
DEVSTRAL_LOG_PATH = "/kaggle/working/devstral-vllm.log"

devstral_server_process = None

def _server_payload():
    try:
        with urllib.request.urlopen(DEVSTRAL_HEALTH_URL, timeout=4) as response:
            return json.loads(response.read().decode("utf-8"))
    except Exception:
        return None

def _devstral_server_ready():
    data = _server_payload()
    return bool(data) and any(x.get("id") == MODEL_ID for x in data.get("data", []))

def load_devstral_model():
    global devstral_server_process
    if _devstral_server_ready():
        return

    vllm_bin = shutil.which("vllm")
    if not vllm_bin:
        raise RuntimeError("vLLM not found. Run Cell 1 again.")

    if torch.cuda.device_count() < 2:
        raise RuntimeError("Devstral requires 2 GPUs on Kaggle.")

    # Removed the unsupported --disable-log-requests flag from the command
    cmd = [
        vllm_bin, "serve", MODEL_ID, "--host", DEVSTRAL_HOST, "--port", str(DEVSTRAL_PORT),
        "--tensor-parallel-size", "2", "--tokenizer-mode", "mistral", "--max-model-len", "8192",
        "--gpu-memory-utilization", "0.80", "--served-model-name", MODEL_ID, "--enforce-eager"
    ]

    print("🚀 Starting local Devstral Small 2 (Will download weights if not cached)...")
    with open(DEVSTRAL_LOG_PATH, "w", encoding="utf-8") as log_file:
        devstral_server_process = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True)

    deadline = time.time() + 1800
    while time.time() < deadline:
        if _devstral_server_ready():
            print("✅ Devstral server READY")
            return
        if devstral_server_process.poll() is not None:
            raise RuntimeError("vLLM server crashed. Check devstral-vllm.log")
        time.sleep(5)
    raise TimeoutError("Timed out waiting for Devstral server.")

print("✅ Devstral Server Manager Loaded")

✅ Devstral Server Manager Loaded


In [4]:
#Framework Generators

import re
import json
from pathlib import Path

def clean_code_block(raw_text, language):
    if not raw_text: return ""
    text = raw_text.strip()
    aliases = {"html": r"html?", "jsx": r"(?:jsx|tsx|javascript|react)"}
    lang = aliases.get(language, re.escape(language))
    m = re.search(rf"```\s*{lang}\s*([\s\S]*?)\s*```", text, re.IGNORECASE)
    if m: return m.group(1).strip()
    m = re.search(r"```\s*([\s\S]*?)\s*```", text)
    return m.group(1).strip() if m else text

class FigmaGenerator:
    @staticmethod
    def generate_plugin(ui_json: dict, out_dir: str):
        figma_dir = os.path.join(out_dir, "figma_plugin")
        os.makedirs(figma_dir, exist_ok=True)
        with open(os.path.join(figma_dir, "manifest.json"), "w") as f:
            json.dump({"name": f"Import UI", "id": "ui-converter", "api": "1.0.0", "main": "code.js", "ui": "ui.html"}, f)
        with open(os.path.join(figma_dir, "ui.json"), "w") as f:
            json.dump(ui_json, f)
        with open(os.path.join(figma_dir, "ui.html"), "w") as f:
            f.write(f"<h2>Figma UI Builder</h2><button id='build'>Build</button><script>document.getElementById('build').onclick = () => parent.postMessage({{pluginMessage:{{type:'generate', data:{json.dumps(ui_json)}}}}}, '*');</script>")

print("✅ Generators Loaded")

✅ Generators Loaded


In [5]:
# ============================================================
# Local Devstral Small 2 24B Multimodal Backend & Server
# ============================================================
import os
import re
import json
import time
import base64
import mimetypes
import shutil
import subprocess
import urllib.request
from pathlib import Path
from huggingface_hub import snapshot_download

# --- CRITICAL KAGGLE BYPASSES ---
os.environ["VLLM_USE_V1"] = "0"              # Force stable V0 engine
os.environ["NCCL_P2P_DISABLE"] = "1"         # Prevent T4 multi-GPU crashes
os.environ["NCCL_SHM_DISABLE"] = "1"         # Bypass Kaggle shared memory limits
os.environ["HF_HOME"] = "/tmp/hf-cache"      # Route massive downloads to the unrestricted /tmp drive
# --------------------------------

import torch
from openai import OpenAI

MODEL_ID = "mistralai/Devstral-Small-2-24B-Instruct-2512"
DEVSTRAL_HOST = "127.0.0.1"
DEVSTRAL_PORT = 8000
DEVSTRAL_BASE_URL = f"http://{DEVSTRAL_HOST}:{DEVSTRAL_PORT}/v1"
DEVSTRAL_HEALTH_URL = f"http://{DEVSTRAL_HOST}:{DEVSTRAL_PORT}/v1/models"
DEVSTRAL_LOG_PATH = "/kaggle/working/devstral-vllm.log"
DEVSTRAL_START_TIMEOUT = 1800
DEVSTRAL_MAX_NEW_TOKENS = 4096
DEVSTRAL_CONTEXT = 4096

LOCAL_MODEL_DIR = "/tmp/devstral_model"
devstral_server_process = None


def _server_payload():
    try:
        with urllib.request.urlopen(DEVSTRAL_HEALTH_URL, timeout=4) as response:
            return json.loads(response.read().decode("utf-8"))
    except Exception:
        return None


def _devstral_server_ready():
    data = _server_payload()
    if not data:
        return False
    return any(x.get("id") == MODEL_ID for x in data.get("data", []))


def _tail_log(lines=80):
    p = Path(DEVSTRAL_LOG_PATH)
    if not p.exists():
        return "<server log unavailable>"
    return "\n".join(p.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:])


def _stop_previous_server():
    try:
        subprocess.run(
            ["bash", "-lc", "pkill -f 'vllm serve' || true"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=10,
        )
    except Exception:
        pass


def patch_and_download_model():
    """Downloads model and physically removes the unsupported FP8 config flag."""
    if not os.path.exists(LOCAL_MODEL_DIR):
        print("📥 Downloading model to local /tmp directory to patch FP8 bug...")
        snapshot_download(
            repo_id=MODEL_ID,
            local_dir=LOCAL_MODEL_DIR,
            ignore_patterns=["*.pt", "*.bin"] # Prioritize safetensors to save space
        )
        
        config_path = os.path.join(LOCAL_MODEL_DIR, "config.json")
        with open(config_path, "r") as f:
            cfg = json.load(f)
            
        if "quantization_config" in cfg:
            print("🔧 Patching config.json: Removing incompatible FP8 hardware flag...")
            del cfg["quantization_config"]
            with open(config_path, "w") as f:
                json.dump(cfg, f, indent=2)


def load_devstral_model(force_restart=False):
    global devstral_server_process

    if _devstral_server_ready() and not force_restart:
        return

    if force_restart:
        _stop_previous_server()
        time.sleep(2)

    vllm_bin = shutil.which("vllm")
    if not vllm_bin:
        raise RuntimeError("vLLM was not found. Re-run your installation cell.")

    gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
    gpu_names = [torch.cuda.get_device_name(i) for i in range(gpu_count)]
    if gpu_count < 2:
        raise RuntimeError(
            f"This local Devstral notebook requires 2 GPUs. Found {gpu_count}: {gpu_names}."
        )

    Path(DEVSTRAL_LOG_PATH).parent.mkdir(parents=True, exist_ok=True)
    patch_and_download_model()

    cmd = [
        vllm_bin, "serve", LOCAL_MODEL_DIR,
        "--host", DEVSTRAL_HOST,
        "--port", str(DEVSTRAL_PORT),
        "--tensor-parallel-size", "2",
        "--tokenizer-mode", "mistral",
        "--max-model-len", "4096",          
        "--max-num-seqs", "1",
        "--max-num-batched-tokens", "4096",
        "--gpu-memory-utilization", "0.80", 
        "--served-model-name", MODEL_ID,    # Keep original ID for API compatibility
        "--enforce-eager",
        "--disable-custom-all-reduce",
        "--quantization", "bitsandbytes",   # Force T4-compatible on-the-fly compression
    ]

    print("CUDA devices:", gpu_count, gpu_names)
    print("🚀 Starting local Devstral Small 2 (T4-Compatible 8-bit Engine)...")
    print("   Model:", MODEL_ID)
    print("   Tensor parallel: 2 GPUs")
    print("   Log:", DEVSTRAL_LOG_PATH)

    with open(DEVSTRAL_LOG_PATH, "w", encoding="utf-8", buffering=1) as log_file:
        devstral_server_process = subprocess.Popen(
            cmd,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            env=os.environ.copy(),
        )

    deadline = time.time() + DEVSTRAL_START_TIMEOUT
    last_report = 0
    while time.time() < deadline:
        if _devstral_server_ready():
            print("✅ Devstral server READY")
            return

        if devstral_server_process.poll() is not None:
            raise RuntimeError(
                "Devstral vLLM server exited during startup.\n\n" + _tail_log()
            )

        now = time.time()
        if now - last_report >= 15:
            print("⏳ Waiting for model load and vLLM startup...")
            last_report = now
        time.sleep(3)

    raise TimeoutError(
        "Timed out waiting for Devstral vLLM server.\n\n" + _tail_log()
    )


def _devstral_client():
    load_devstral_model()
    return OpenAI(base_url=DEVSTRAL_BASE_URL, api_key="local", timeout=1800.0)


def _image_data_uri(image_path):
    path = Path(image_path)
    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}")
    mime = mimetypes.guess_type(path.name)[0] or "image/png"
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime};base64,{encoded}"


def query_devstral(prompt, image_path=None, max_new_tokens=DEVSTRAL_MAX_NEW_TOKENS, temperature=0.15):
    if not prompt or not prompt.strip():
        raise ValueError("Prompt cannot be empty.")

    client = _devstral_client()
    content = prompt
    if image_path:
        content = [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": _image_data_uri(image_path)}},
        ]

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": "You are Devstral, a precise software-engineering and UI reconstruction model."},
            {"role": "user", "content": content},
        ],
        temperature=temperature,
        max_tokens=max_new_tokens,
    )
    text = response.choices[0].message.content
    if not text or not text.strip():
        raise RuntimeError("Devstral returned an empty response.")
    return text.strip()


def query_devstral_vision(image_path, prompt):
    return query_devstral(prompt=prompt, image_path=image_path)


def query_devstral_text(prompt):
    return query_devstral(prompt=prompt)


def query_vision_model(image_path, prompt, engine="devstral_24b"):
    if str(engine).strip().lower() != "devstral_24b":
        raise ValueError("This notebook is local-only; use Devstral 24B.")
    print("Vision request -> devstral_24b")
    return query_devstral_vision(image_path, prompt)


def query_text_model(prompt, engine="devstral_24b"):
    if str(engine).strip().lower() != "devstral_24b":
        raise ValueError("This notebook is local-only; use Devstral 24B.")
    print("Text request -> devstral_24b")
    return query_devstral_text(prompt)


def clean_code_block(raw_text, language):
    if not raw_text:
        return ""
    text = raw_text.strip()
    aliases = {"html": r"html?", "jsx": r"(?:jsx|tsx|javascript|react)"}
    lang = aliases.get(language, re.escape(language))
    m = re.search(rf"```\s*{lang}\s*([\s\S]*?)\s*```", text, re.IGNORECASE)
    if m:
        return m.group(1).strip()
    m = re.search(r"```\s*([\s\S]*?)\s*```", text)
    return m.group(1).strip() if m else text


class HTMLGenerator:
    @staticmethod
    def generate(image_path, out_dir, engine="devstral_24b"):
        html_dir = Path(out_dir) / "html"
        html_dir.mkdir(parents=True, exist_ok=True)
        prompt = """
You are a principal frontend engineer specializing in screenshot-to-code reconstruction.
Analyze the supplied screenshot and create a highly accurate responsive HTML5 implementation.

Return ONLY the complete HTML document.

Requirements:
- Start with <!DOCTYPE html>.
- Match geometry, spacing, colors, typography, borders, shadows, radii and alignment closely.
- Reproduce readable text exactly.
- Use semantic HTML and accessible controls.
- Put CSS in the document.
- Use inline SVG/CSS/simple placeholders instead of inventing external asset URLs.
- Make it standalone and responsive.
- Do not use Markdown fences or explanations.
"""
        print("Generating HTML via [DEVSTRAL_24B]...")
        raw = query_vision_model(image_path, prompt, engine)
        code = clean_code_block(raw, "html")
        if "<html" not in code.lower() and "<!doctype" not in code.lower():
            raise RuntimeError("Devstral did not return a complete HTML document.")
        path = html_dir / "index.html"
        path.write_text(code, encoding="utf-8")
        return str(path)


class ReactTailwindGenerator:
    @staticmethod
    def generate(image_path, out_dir, engine="devstral_24b"):
        root = Path(out_dir) / "react"
        src = root / "src"
        src.mkdir(parents=True, exist_ok=True)
        prompt = """
You are an expert React frontend engineer. Recreate the supplied screenshot as a pixel-accurate responsive React + Tailwind UI.

Return ONLY App.jsx. No Markdown fences. No explanation.

Requirements:
- Use export default function App() { ... }.
- Use semantic, accessible JSX.
- Match layout, dimensions, spacing, typography, colors, borders, shadows and radii.
- Reproduce readable text exactly.
- Do not rely on invented external image URLs for core visuals.
- Keep the component self-contained and Vite-compatible.
"""
        print("Generating React via [DEVSTRAL_24B]...")
        raw = query_vision_model(image_path, prompt, engine)
        jsx = clean_code_block(raw, "jsx")
        if "function App" not in jsx and "export default" not in jsx:
            raise RuntimeError("Devstral did not return a recognizable React App component.")

        (src / "App.jsx").write_text(jsx, encoding="utf-8")
        (root / "package.json").write_text(json.dumps({
            "name": "screenshot-to-ui-react",
            "private": True,
            "version": "1.0.0",
            "type": "module",
            "scripts": {"dev": "vite", "build": "vite build", "preview": "vite preview"},
            "dependencies": {"react": "^18.3.1", "react-dom": "^18.3.1"},
            "devDependencies": {"@vitejs/plugin-react": "^4.3.1", "tailwindcss": "^3.4.1", "vite": "^5.4.2"}
        }, indent=2), encoding="utf-8")
        (root / "vite.config.js").write_text(
            "import { defineConfig } from 'vite'\nimport react from '@vitejs/plugin-react'\nexport default defineConfig({plugins:[react()]})\n",
            encoding="utf-8"
        )
        (root / "index.html").write_text(
            "<!doctype html><html lang='en'><head><meta charset='UTF-8'><meta name='viewport' content='width=device-width,initial-scale=1.0'><title>Screenshot UI</title></head><body><div id='root'></div><script type='module' src='/src/main.jsx'></script></body></html>",
            encoding="utf-8"
        )
        (src / "main.jsx").write_text(
            "import React from 'react';\nimport ReactDOM from 'react-dom/client';\nimport App from './App';\nimport './index.css';\nReactDOM.createRoot(document.getElementById('root')).render(<React.StrictMode><App /></React.StrictMode>);\n",
            encoding="utf-8"
        )
        (src / "index.css").write_text(
            "@tailwind base;\n@tailwind components;\n@tailwind utilities;\nhtml,body,#root{min-height:100%;}body{margin:0;}\n",
            encoding="utf-8"
        )
        (root / "tailwind.config.js").write_text(
            "export default {content:['./index.html','./src/**/*.{js,jsx,ts,tsx}'],theme:{extend:{}},plugins:[]};\n",
            encoding="utf-8"
        )
        return str(root)


print("✅ Local Devstral 24B router/generators loaded with hardware bypasses applied")

✅ Local Devstral 24B router/generators loaded with hardware bypasses applied


In [6]:
# ============================================================
# Local Devstral Small 2 24B Multimodal Backend & Server
# ============================================================
import os
import re
import json
import time
import base64
import mimetypes
import shutil
import subprocess
import urllib.request
import glob
from pathlib import Path
from huggingface_hub import snapshot_download

import torch
from openai import OpenAI

MODEL_ID = "mistralai/Devstral-Small-2-24B-Instruct-2512"
GGUF_REPO = "byteshape/Devstral-Small-2-24B-Instruct-2512-GGUF"
DEVSTRAL_HOST = "127.0.0.1"
DEVSTRAL_PORT = 8000
DEVSTRAL_BASE_URL = f"http://{DEVSTRAL_HOST}:{DEVSTRAL_PORT}/v1"
DEVSTRAL_HEALTH_URL = f"http://{DEVSTRAL_HOST}:{DEVSTRAL_PORT}/v1/models"
DEVSTRAL_LOG_PATH = "/kaggle/working/devstral-llamacpp.log"
DEVSTRAL_START_TIMEOUT = 1800
DEVSTRAL_MAX_NEW_TOKENS = 4096
DEVSTRAL_CONTEXT = 4096

LOCAL_MODEL_DIR = "/tmp/devstral_gguf"
devstral_server_process = None


def _server_payload():
    try:
        with urllib.request.urlopen(DEVSTRAL_HEALTH_URL, timeout=4) as response:
            return json.loads(response.read().decode("utf-8"))
    except Exception:
        return None


def _devstral_server_ready():
    data = _server_payload()
    if not data:
        return False
    return any(x.get("id") == MODEL_ID for x in data.get("data", []))


def _tail_log(lines=80):
    p = Path(DEVSTRAL_LOG_PATH)
    if not p.exists():
        return "<server log unavailable>"
    return "\n".join(p.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:])


def _stop_previous_server():
    try:
        subprocess.run(
            ["bash", "-lc", "pkill -f 'llama-server' || true"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=10,
        )
    except Exception:
        pass


def download_gguf_model():
    """Downloads the <=15GB quantized model and vision projector."""
    os.environ["HF_HOME"] = "/tmp/hf-cache"
    print(f"📥 Downloading <=15GB Quantized Model from {GGUF_REPO}...")
    model_dir = snapshot_download(
        repo_id=GGUF_REPO,
        allow_patterns=["*IQ4_XS*.gguf", "mmproj*.gguf"],
        local_dir=LOCAL_MODEL_DIR
    )
    
    model_paths = glob.glob(os.path.join(model_dir, "*IQ4_XS*.gguf"))
    mmproj_paths = glob.glob(os.path.join(model_dir, "mmproj*.gguf"))
    
    if not model_paths or not mmproj_paths:
        raise RuntimeError("Failed to find the required GGUF files after download.")
        
    return model_paths[0], mmproj_paths[0]


def load_devstral_model(force_restart=False):
    global devstral_server_process

    if _devstral_server_ready() and not force_restart:
        return

    if force_restart:
        _stop_previous_server()
        time.sleep(2)

    llama_server_bin = "/kaggle/working/llama.cpp/build/bin/llama-server"
    if not os.path.isfile(llama_server_bin):
        raise RuntimeError("llama-server binary not found! Please run Cell 7 to build llama.cpp first.")

    Path(DEVSTRAL_LOG_PATH).parent.mkdir(parents=True, exist_ok=True)
    
    model_path, mmproj_path = download_gguf_model()

    cmd = [
        llama_server_bin,
        "-m", model_path,
        "--mmproj", mmproj_path,
        "--host", DEVSTRAL_HOST,
        "--port", str(DEVSTRAL_PORT),
        "--alias", MODEL_ID,         # Ensures API compatibility with the UI
        "-c", "4096",                # Context window
        "-ngl", "99",                # Offload all layers to GPU
        "-ts", "1,1",                # Split evenly across 2 GPUs
        "--parallel", "1",
    ]

    print("🚀 Starting local Devstral Small 2 (llama.cpp GGUF Backend)...")
    print("   Model:", os.path.basename(model_path))
    print("   Vision:", os.path.basename(mmproj_path))
    print("   Log:", DEVSTRAL_LOG_PATH)

    with open(DEVSTRAL_LOG_PATH, "w", encoding="utf-8", buffering=1) as log_file:
        devstral_server_process = subprocess.Popen(
            cmd,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            env=os.environ.copy(),
        )

    deadline = time.time() + DEVSTRAL_START_TIMEOUT
    last_report = 0
    while time.time() < deadline:
        if _devstral_server_ready():
            print("✅ Devstral server READY")
            return

        if devstral_server_process.poll() is not None:
            raise RuntimeError(
                "Devstral llama-server exited during startup.\n\n" + _tail_log()
            )

        now = time.time()
        if now - last_report >= 15:
            print("⏳ Waiting for GGUF model load and llama-server startup...")
            last_report = now
        time.sleep(3)

    raise TimeoutError(
        "Timed out waiting for Devstral llama-server.\n\n" + _tail_log()
    )


def _devstral_client():
    load_devstral_model()
    return OpenAI(base_url=DEVSTRAL_BASE_URL, api_key="local", timeout=1800.0)


def _image_data_uri(image_path):
    path = Path(image_path)
    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}")
    mime = mimetypes.guess_type(path.name)[0] or "image/png"
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime};base64,{encoded}"


def query_devstral(prompt, image_path=None, max_new_tokens=DEVSTRAL_MAX_NEW_TOKENS, temperature=0.15):
    if not prompt or not prompt.strip():
        raise ValueError("Prompt cannot be empty.")

    client = _devstral_client()
    content = prompt
    if image_path:
        content = [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": _image_data_uri(image_path)}},
        ]

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": "You are Devstral, a precise software-engineering and UI reconstruction model."},
            {"role": "user", "content": content},
        ],
        temperature=temperature,
        max_tokens=max_new_tokens,
    )
    text = response.choices[0].message.content
    if not text or not text.strip():
        raise RuntimeError("Devstral returned an empty response.")
    return text.strip()


def query_devstral_vision(image_path, prompt):
    return query_devstral(prompt=prompt, image_path=image_path)


def query_devstral_text(prompt):
    return query_devstral(prompt=prompt)


def query_vision_model(image_path, prompt, engine="devstral_24b"):
    if str(engine).strip().lower() != "devstral_24b":
        raise ValueError("This notebook is local-only; use Devstral 24B.")
    print("Vision request -> devstral_24b")
    return query_devstral_vision(image_path, prompt)


def query_text_model(prompt, engine="devstral_24b"):
    if str(engine).strip().lower() != "devstral_24b":
        raise ValueError("This notebook is local-only; use Devstral 24B.")
    print("Text request -> devstral_24b")
    return query_devstral_text(prompt)


def clean_code_block(raw_text, language):
    if not raw_text:
        return ""
    text = raw_text.strip()
    aliases = {"html": r"html?", "jsx": r"(?:jsx|tsx|javascript|react)"}
    lang = aliases.get(language, re.escape(language))
    m = re.search(rf"```\s*{lang}\s*([\s\S]*?)\s*```", text, re.IGNORECASE)
    if m:
        return m.group(1).strip()
    m = re.search(r"```\s*([\s\S]*?)\s*```", text)
    return m.group(1).strip() if m else text


class HTMLGenerator:
    @staticmethod
    def generate(image_path, out_dir, engine="devstral_24b"):
        html_dir = Path(out_dir) / "html"
        html_dir.mkdir(parents=True, exist_ok=True)
        prompt = """
You are a principal frontend engineer specializing in screenshot-to-code reconstruction.
Analyze the supplied screenshot and create a highly accurate responsive HTML5 implementation.

Return ONLY the complete HTML document.

Requirements:
- Start with <!DOCTYPE html>.
- Match geometry, spacing, colors, typography, borders, shadows, radii and alignment closely.
- Reproduce readable text exactly.
- Use semantic HTML and accessible controls.
- Put CSS in the document.
- Use inline SVG/CSS/simple placeholders instead of inventing external asset URLs.
- Make it standalone and responsive.
- Do not use Markdown fences or explanations.
"""
        print("Generating HTML via [DEVSTRAL_24B]...")
        raw = query_vision_model(image_path, prompt, engine)
        code = clean_code_block(raw, "html")
        if "<html" not in code.lower() and "<!doctype" not in code.lower():
            raise RuntimeError("Devstral did not return a complete HTML document.")
        path = html_dir / "index.html"
        path.write_text(code, encoding="utf-8")
        return str(path)


class ReactTailwindGenerator:
    @staticmethod
    def generate(image_path, out_dir, engine="devstral_24b"):
        root = Path(out_dir) / "react"
        src = root / "src"
        src.mkdir(parents=True, exist_ok=True)
        prompt = """
You are an expert React frontend engineer. Recreate the supplied screenshot as a pixel-accurate responsive React + Tailwind UI.

Return ONLY App.jsx. No Markdown fences. No explanation.

Requirements:
- Use export default function App() { ... }.
- Use semantic, accessible JSX.
- Match layout, dimensions, spacing, typography, colors, borders, shadows and radii.
- Reproduce readable text exactly.
- Do not rely on invented external image URLs for core visuals.
- Keep the component self-contained and Vite-compatible.
"""
        print("Generating React via [DEVSTRAL_24B]...")
        raw = query_vision_model(image_path, prompt, engine)
        jsx = clean_code_block(raw, "jsx")
        if "function App" not in jsx and "export default" not in jsx:
            raise RuntimeError("Devstral did not return a recognizable React App component.")

        (src / "App.jsx").write_text(jsx, encoding="utf-8")
        (root / "package.json").write_text(json.dumps({
            "name": "screenshot-to-ui-react",
            "private": True,
            "version": "1.0.0",
            "type": "module",
            "scripts": {"dev": "vite", "build": "vite build", "preview": "vite preview"},
            "dependencies": {"react": "^18.3.1", "react-dom": "^18.3.1"},
            "devDependencies": {"@vitejs/plugin-react": "^4.3.1", "tailwindcss": "^3.4.1", "vite": "^5.4.2"}
        }, indent=2), encoding="utf-8")
        (root / "vite.config.js").write_text(
            "import { defineConfig } from 'vite'\nimport react from '@vitejs/plugin-react'\nexport default defineConfig({plugins:[react()]})\n",
            encoding="utf-8"
        )
        (root / "index.html").write_text(
            "<!doctype html><html lang='en'><head><meta charset='UTF-8'><meta name='viewport' content='width=device-width,initial-scale=1.0'><title>Screenshot UI</title></head><body><div id='root'></div><script type='module' src='/src/main.jsx'></script></body></html>",
            encoding="utf-8"
        )
        (src / "main.jsx").write_text(
            "import React from 'react';\nimport ReactDOM from 'react-dom/client';\nimport App from './App';\nimport './index.css';\nReactDOM.createRoot(document.getElementById('root')).render(<React.StrictMode><App /></React.StrictMode>);\n",
            encoding="utf-8"
        )
        (src / "index.css").write_text(
            "@tailwind base;\n@tailwind components;\n@tailwind utilities;\nhtml,body,#root{min-height:100%;}body{margin:0;}\n",
            encoding="utf-8"
        )
        (root / "tailwind.config.js").write_text(
            "export default {content:['./index.html','./src/**/*.{js,jsx,ts,tsx}'],theme:{extend:{}},plugins:[]};\n",
            encoding="utf-8"
        )
        return str(root)


print("✅ Local Devstral 24B router/generators loaded with llama.cpp Backend")

✅ Local Devstral 24B router/generators loaded with llama.cpp Backend


In [7]:
# ============================================================
# Build llama.cpp CUDA server for Kaggle 2×T4
# ============================================================

import os
import shutil
import subprocess

LLAMA_SRC = "/kaggle/working/llama.cpp"
LLAMA_BUILD = os.path.join(LLAMA_SRC, "build")
LLAMA_SERVER_BIN = os.path.join(LLAMA_BUILD, "bin", "llama-server")

print("=" * 72)
print("🦙 LLAMA.CPP — CUDA BUILD FOR 2×T4")
print("=" * 72)

# 1. CHECK GPU
print("\n[1/5] CHECKING NVIDIA GPU")
gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    raise RuntimeError("❌ NVIDIA GPU/CUDA is not available.")
print("✅ NVIDIA GPU detected")

# 2. CHECK SOURCE
print("\n[2/5] CHECKING LLAMA.CPP SOURCE")
if not os.path.isdir(LLAMA_SRC):
    print("⏳ llama.cpp not found. Cloning repository...")
    subprocess.run([
        "git", "clone", "--depth", "1", 
        "https://github.com/ggml-org/llama.cpp.git", LLAMA_SRC
    ], check=True)
else:
    print("✅ llama.cpp source exists")

# 3. CLEAN OLD BUILD
print("\n[3/5] CLEANING PREVIOUS BUILD")
if os.path.isdir(LLAMA_BUILD):
    shutil.rmtree(LLAMA_BUILD, ignore_errors=True)
print("✅ Clean build directory prepared")

# 4. CONFIGURE CUDA
print("\n[4/5] CONFIGURING CUDA (Tesla T4)")
configure_cmd = [
    "cmake", "-S", LLAMA_SRC, "-B", LLAMA_BUILD,
    "-DGGML_CUDA=ON", 
    "-DGGML_CUDA_NO_VMM=ON", 
    "-DCMAKE_CUDA_ARCHITECTURES=75", # Architecture for Tesla T4
    "-DGGML_NATIVE=OFF", 
    "-DLLAMA_CURL=OFF", 
    "-DCMAKE_BUILD_TYPE=Release",
]
subprocess.run(configure_cmd, check=True)
print("✅ CUDA configuration successful")

# 5. COMPILE
print("\n[5/5] BUILDING LLAMA-SERVER (This takes a few minutes...)")
build_cmd = [
    "cmake", "--build", LLAMA_BUILD, 
    "--config", "Release", 
    "--target", "llama-server", 
    "-j", "2"
]
subprocess.run(build_cmd, check=True)

# VERIFY
if not os.path.isfile(LLAMA_SERVER_BIN):
    raise RuntimeError(f"❌ Build finished but binary was not found at {LLAMA_SERVER_BIN}")

os.chmod(LLAMA_SERVER_BIN, 0o755)

print("\n" + "=" * 72)
print("✅ LLAMA.CPP BUILD SUCCESSFUL")
print("=" * 72)

🦙 LLAMA.CPP — CUDA BUILD FOR 2×T4

[1/5] CHECKING NVIDIA GPU
✅ NVIDIA GPU detected

[2/5] CHECKING LLAMA.CPP SOURCE
✅ llama.cpp source exists

[3/5] CLEANING PREVIOUS BUILD
✅ Clean build directory prepared

[4/5] CONFIGURING CUDA (Tesla T4)
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.4.1-dev
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD


CMAKE_BUILD_TYPE=Release


-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found OpenMP_C: -fopenmp (found version "4.5")
-- Found OpenMP_CXX: -fopenmp (found version "4.5")
-- Found OpenMP: TRUE (found version "4.5")
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -msse4.2;-mf16c;-mfma;-mbmi2;-mavx;-mavx2 GGML_SSE42;GGML_F16C;GGML_FMA;GGML_BMI2;GGML_AVX;GGML_AVX2
-- Found CUDAToolkit: /usr/local/cuda/targets/x86_64-linux/include (found version "12.8.93")
-- CUDA Toolkit found
-- The CUDA compiler identification is NVIDIA 12.8.93 with host compiler GNU 11.4.0
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile featu

In static member function ‘static _Tp* std::__copy_move<_IsMove, true, std::random_access_iterator_tag>::__copy_m(const _Tp*, const _Tp*, _Tp*) [with _Tp = ggml_op; bool _IsMove = false]’,
    inlined from ‘_OI std::__copy_move_a2(_II, _II, _OI) [with bool _IsMove = false; _II = const ggml_op*; _OI = ggml_op*]’ at /usr/include/c++/11/bits/stl_algobase.h:494:141,
    inlined from ‘_OI std::__copy_move_a1(_II, _II, _OI) [with bool _IsMove = false; _II = const ggml_op*; _OI = ggml_op*]’ at /usr/include/c++/11/bits/stl_algobase.h:522:40,
    inlined from ‘_OI std::__copy_move_a(_II, _II, _OI) [with bool _IsMove = false; _II = const ggml_op*; _OI = ggml_op*]’ at /usr/include/c++/11/bits/stl_algobase.h:529:25,
    inlined from ‘_OI std::copy(_II, _II, _OI) [with _II = const ggml_op*; _OI = ggml_op*]’ at /usr/include/c++/11/bits/stl_algobase.h:619:64,
    inlined from ‘static _ForwardIterator std::__uninitialized_copy<true>::__uninit_copy(_InputIterator, _InputIterator, _ForwardIterator) [wit

[ 20%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/lightning-indexer.cu.o
[ 20%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/mean.cu.o
[ 20%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/mmf.cu.o
[ 20%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/mmid.cu.o
[ 22%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/mmq.cu.o
[ 22%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/mmvf.cu.o
[ 22%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/mmvq.cu.o
[ 22%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/moe-weighted-reduction.cu.o
[ 22%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/norm.cu.o
[ 22%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/opt-step-adamw.cu.o
[ 24%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/opt-step-sgd.cu.o
[ 24%] Building CUDA object ggml/src/ggml-cuda/CMak

In [8]:
# ============================================================
# CELL 12 - PROJECT AI ASSISTANT (CHATBOT)
# Conversational interface connected to your generated project
# ============================================================

import os
import json
import requests
import html
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Ensure we use the API from previous cells
CHAT_API = globals().get("DEVSTRAL_BASE_URL", "http://127.0.0.1:8000/v1")
CHAT_MODEL = globals().get("MODEL_ID", "mistralai/Devstral-Small-2-24B-Instruct-2512")
CHAT_PROJECT_ROOT = globals().get("PROJECT_ROOT", "/kaggle/working/generated_project")

# ------------------------------------------------------------
# 1. UI COMPONENTS
# ------------------------------------------------------------

chat_header = widgets.HTML(
    """
    <div style="
        margin:20px 0 14px;
        padding:19px;
        border-radius:11px;
        background:linear-gradient(135deg, #0f172a, #4338ca);
        color:white;
    ">
        <h2 style="margin:0 0 6px;">💬 Project AI Assistant</h2>
        <div style="font-size:13px; opacity:.92;">
            Chat with Devstral about your generated project. It knows your file structure.
        </div>
    </div>
    """
)

chat_display = widgets.Output(
    layout=widgets.Layout(
        width="100%",
        height="400px",
        border="1px solid #cbd5e1",
        border_radius="10px",
        overflow_y="auto",
        padding="15px",
        margin="0 0 15px 0"
    )
)

chat_input = widgets.Text(
    placeholder="Ask a question or request a code snippet...",
    layout=widgets.Layout(width="80%", height="45px"),
    continuous_update=False  # Fixes the deprecation warning
)

send_button = widgets.Button(
    description="Send",
    button_style="primary",
    layout=widgets.Layout(width="18%", height="32px", margin="0 0 0 2%")
)

clear_chat_button = widgets.Button(
    description="🗑️ Clear Chat",
    button_style="warning",
    layout=widgets.Layout(width="150px", margin="10px 0 0 0")
)

# ------------------------------------------------------------
# 2. CHAT STATE & LOGIC
# ------------------------------------------------------------

conversation_history = []

def format_message(role, text):
    if role == "user":
        return f"""
        <div style="margin-bottom: 15px; display: flex; justify-content: flex-end;">
            <div style="background-color: #dbeafe; color: #1e3a8a; padding: 12px 16px; border-radius: 12px 12px 0 12px; max-width: 80%; font-family: system-ui; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
                <b>You</b><br>{html.escape(text).replace(chr(10), '<br>')}
            </div>
        </div>
        """
    else:
        # Simple markdown code block formatter for the UI
        formatted_text = html.escape(text).replace("```", "<hr style='border-color:#cbd5e1;'>").replace(chr(10), '<br>')
        return f"""
        <div style="margin-bottom: 15px; display: flex; justify-content: flex-start;">
            <div style="background-color: #f8fafc; color: #334155; padding: 12px 16px; border-radius: 12px 12px 12px 0; border: 1px solid #e2e8f0; max-width: 85%; font-family: system-ui; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
                <b>🤖 Devstral</b><br>{formatted_text}
            </div>
        </div>
        """

def get_project_context():
    """Grabs the current file tree to give the AI context."""
    if 'project_files' in globals():
        files = globals()['project_files']()
        return "\n".join(files) if files else "(Project is empty)"
    return "(Project file structure unavailable)"

def handle_send(*args, **kwargs):
    user_text = chat_input.value.strip()
    if not user_text:
        return
    
    # 1. Update UI with User Message
    chat_input.value = ""
    send_button.disabled = True
    send_button.description = "Thinking..."
    
    conversation_history.append({"role": "user", "content": user_text})
    
    with chat_display:
        display(HTML(format_message("user", user_text)))
    
    # 2. Build Prompt with Context
    system_prompt = (
        "You are an expert software engineer assistant. "
        "The user has just generated a frontend UI project using your multimodal capabilities. "
        f"Here is the current file structure of their project located at {CHAT_PROJECT_ROOT}:\n\n"
        f"{get_project_context()}\n\n"
        "Answer their questions concisely, provide code snippets, or explain how to modify the project. "
        "Do NOT output JSON manifests unless explicitly asked."
    )
    
    messages = [{"role": "system", "content": system_prompt}] + conversation_history
    
    # 3. Call Devstral
    try:
        # THE FIX: Ensure server is running before hitting the API
        if 'load_devstral_model' in globals():
            globals()['load_devstral_model']()
            
        response = requests.post(
            f"{CHAT_API}/chat/completions",
            json={
                "model": CHAT_MODEL,
                "messages": messages,
                "temperature": 0.3,
                "max_tokens": 2048
            },
            timeout=120
        )
        response.raise_for_status()
        
        ai_reply = response.json()["choices"][0]["message"]["content"].strip()
        conversation_history.append({"role": "assistant", "content": ai_reply})
        
        with chat_display:
            display(HTML(format_message("assistant", ai_reply)))
            
    except Exception as e:
        with chat_display:
            display(HTML(f"<div style='color: red; padding: 10px;'><b>Error connecting to Devstral:</b> {e}</div>"))
            
    finally:
        send_button.disabled = False
        send_button.description = "Send"

def clear_chat(_):
    global conversation_history
    conversation_history = []
    with chat_display:
        clear_output()
        display(HTML("<div style='color: #64748b; font-style: italic; text-align: center; margin-top: 20px;'>Chat history cleared. Context refreshed.</div>"))

# ------------------------------------------------------------
# 3. EVENT BINDING & DISPLAY
# ------------------------------------------------------------

send_button.on_click(handle_send)
chat_input.observe(handle_send, names='value') # Modern implementation replacing on_submit
clear_chat_button.on_click(clear_chat)

display(chat_header)
display(chat_display)
display(widgets.HBox([chat_input, send_button], layout=widgets.Layout(align_items="center")))
display(clear_chat_button)

# Initialize display
with chat_display:
    display(HTML("<div style='color: #64748b; font-style: italic; text-align: center; margin-top: 20px;'>Ask me how to customize your new UI, what packages to install, or to write new components for you!</div>"))

HTML(value='\n    <div style="\n        margin:20px 0 14px;\n        padding:19px;\n        border-radius:11px…

Output(layout=Layout(border_bottom='1px solid #cbd5e1', border_left='1px solid #cbd5e1', border_right='1px sol…

Button(button_style='warning', description='🗑️ Clear Chat', layout=Layout(margin='10px 0 0 0', width='150px'),…

In [9]:
# ============================================================
# COMPLETE PROJECT GENERATOR DASHBOARD
# Upload screenshot -> Select Framework -> Generate Code
# ============================================================

import os
import re
import json
import base64
import shutil
import time
import zipfile
import requests
import mimetypes

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output, FileLink

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

# Safely get variables from previous cells
BASE_URL = globals().get("DEVSTRAL_BASE_URL", "http://127.0.0.1:8000/v1")
MODEL = globals().get("MODEL_ID", "mistralai/Devstral-Small-2-24B-Instruct-2512")
PROJECT_ROOT = "/kaggle/working/generated_project"

os.makedirs(PROJECT_ROOT, exist_ok=True)

# ------------------------------------------------------------
# 2. UTILITY FUNCTIONS
# ------------------------------------------------------------

def image_to_data_uri(path):
    if not path or not os.path.exists(path):
        return None
    mime, _ = mimetypes.guess_type(path)
    if not mime:
        mime = "image/png"
    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("ascii")
    return f"data:{mime};base64,{encoded}"

def clean_json_response(text):
    if not text:
        return ""
    text = str(text).strip()
    match = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        return text[start:end + 1].strip()
    return text

def safe_project_path(relative_path):
    relative_path = str(relative_path).replace("\\", "/").strip().lstrip("/")
    parts = [p for p in relative_path.split("/") if p not in {"", ".", ".."}]
    clean = "/".join(parts)
    if not clean:
        raise ValueError("Invalid empty project path.")
    return clean

def write_project_files(project):
    files = project.get("files", [])
    if not isinstance(files, list):
        raise ValueError("AI project manifest does not contain a valid files list.")
    
    written = []
    for item in files:
        if not isinstance(item, dict):
            continue
        path = item.get("path")
        content = item.get("content", "")
        if not path:
            continue
            
        path = safe_project_path(path)
        destination = os.path.join(PROJECT_ROOT, path)
        
        # Security check to prevent directory traversal
        root_abs = os.path.abspath(PROJECT_ROOT)
        dest_abs = os.path.abspath(destination)
        if not (dest_abs == root_abs or dest_abs.startswith(root_abs + os.sep)):
            raise ValueError(f"Unsafe generated path: {path}")
            
        os.makedirs(os.path.dirname(destination), exist_ok=True)
        with open(destination, "w", encoding="utf-8") as f:
            f.write(str(content))
        written.append(destination)
        
    if not written:
        raise ValueError("AI did not generate any project files.")
    return written

def create_zip():
    zip_path = "/kaggle/working/screenshot-ui-project.zip"
    if os.path.exists(zip_path):
        os.remove(zip_path)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for root, dirs, files in os.walk(PROJECT_ROOT):
            for filename in files:
                full_path = os.path.join(root, filename)
                arcname = os.path.relpath(full_path, PROJECT_ROOT)
                z.write(full_path, arcname)
    return zip_path

def project_file_tree():
    result = []
    for root, dirs, files in os.walk(PROJECT_ROOT):
        dirs[:] = [d for d in dirs if d != "__pycache__"]
        for filename in files:
            path = os.path.join(root, filename)
            result.append(os.path.relpath(path, PROJECT_ROOT).replace("\\", "/"))
    return sorted(result)

def get_uploaded_image():
    value = screenshot_upload.value
    if not value:
        return None, None

    # Handle different Kaggle widget formats
    if isinstance(value, (tuple, list)):
        if len(value) == 0: return None, None
        item = value[0]
        if isinstance(item, dict):
            return item.get("name", "screenshot.png"), item.get("content", b"")
            
    if isinstance(value, dict):
        if not value: return None, None
        name, item = next(iter(value.items()))
        if isinstance(item, dict):
            return item.get("name", name), item.get("content", b"")
        return name, item

    return None, None

# ------------------------------------------------------------
# 3. AI API CALL
# ------------------------------------------------------------

def call_devstral_project(screenshot_path, framework, instruction, project_name):
    image_uri = image_to_data_uri(screenshot_path)
    
    system_prompt = """
You are a principal frontend architect and expert screenshot-to-production-project engineer.
You generate COMPLETE software projects. You do NOT generate only one component.
The user's selected framework/stack is mandatory.
Return a machine-readable JSON project manifest.
"""

    user_prompt = f"""
Create a COMPLETE production-ready frontend project from the uploaded screenshot.

Project name: {project_name}
Framework / Stack: {framework}
User instruction: {instruction}

Generate ALL files required for the selected framework/stack.
Do NOT return only one file. Return the COMPLETE PROJECT STRUCTURE.
Use exactly the requested framework/stack. Do not silently replace it.

Reconstruct the layout, colors, typography, spacing, and responsive behavior exactly as seen in the screenshot.

Return EXACTLY this JSON structure:
{{
  "project_name": "string",
  "framework": "string",
  "description": "string",
  "files": [
    {{
      "path": "package.json",
      "content": "FULL FILE CONTENT"
    }},
    {{
      "path": "src/App.tsx",
      "content": "FULL FILE CONTENT"
    }}
  ]
}}

IMPORTANT:
- Every file must contain its COMPLETE content. Never use "...".
- Return valid JSON only. Do not wrap JSON in Markdown fences.
"""

    content = [{"type": "text", "text": user_prompt}]
    if image_uri:
        content.append({"type": "image_url", "image_url": {"url": image_uri}})

    response = requests.post(
        BASE_URL + "/chat/completions",
        json={
            "model": MODEL,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": content}
            ],
            "temperature": 0.08,
            "max_tokens": 30000,
        },
        timeout=1800
    )
    response.raise_for_status()
    
    data = response.json()
    ai_content = data["choices"][0]["message"]["content"]
    
    raw_json = clean_json_response(ai_content)
    try:
        project = json.loads(raw_json)
    except Exception as exc:
        raise RuntimeError(f"Devstral did not return valid JSON.\n\n{exc}\n\n{raw_json[:20000]}")

    return project

# ------------------------------------------------------------
# 4. DASHBOARD WIDGETS
# ------------------------------------------------------------

header = widgets.HTML(
    """
    <div style="margin:20px 0 15px; padding:20px; border-radius:12px; background:linear-gradient(135deg, #111827, #1d4ed8); color:white;">
        <h2 style="margin:0 0 7px;">🎨 Screenshot → Complete Project</h2>
        <div style="font-size:13px; opacity:.92;">Upload a UI screenshot and generate the full project code.</div>
    </div>
    """
)

screenshot_upload = widgets.FileUpload(accept="image/*", multiple=False, description="📷 Upload Screenshot", layout=widgets.Layout(width="260px"))
project_name_input = widgets.Text(value="screenshot-ui", description="Project Name:", layout=widgets.Layout(width="650px"))
framework_input = widgets.Text(value="Next.js + TypeScript + Tailwind CSS", description="Stack:", layout=widgets.Layout(width="850px"))
instruction_input = widgets.Textarea(value="", description="Instruction:", placeholder="E.g., Make it responsive, use dark mode...", layout=widgets.Layout(width="850px", height="100px"))

generate_button = widgets.Button(description="🚀 Generate Complete Project", button_style="success", layout=widgets.Layout(width="260px", height="46px"))
clear_button = widgets.Button(description="🗑️ Clear", button_style="warning", layout=widgets.Layout(width="120px", height="46px"))
output = widgets.Output()

# ------------------------------------------------------------
# 5. EXECUTION LOGIC
# ------------------------------------------------------------

_running = False

def generate_project(_):
    global _running
    if _running: return
    
    framework = framework_input.value.strip()
    instruction = instruction_input.value.strip()
    project_name = project_name_input.value.strip() or "screenshot-ui"

    with output:
        clear_output()
        if not framework:
            print("⚠️ Please enter a framework / stack.")
            return

        filename, image_content = get_uploaded_image()
        if not filename or not image_content:
            print("⚠️ Please upload a screenshot first.")
            return

        # Prepare workspace
        shutil.rmtree(PROJECT_ROOT, ignore_errors=True)
        os.makedirs(PROJECT_ROOT, exist_ok=True)

        input_path = os.path.join("/kaggle/working", f"_ui_reference_{os.path.basename(filename)}")
        
        if isinstance(image_content, memoryview): image_content = image_content.tobytes()
        elif isinstance(image_content, bytearray): image_content = bytes(image_content)
            
        with open(input_path, "wb") as f:
            f.write(image_content)

        _running = True
        generate_button.disabled = True
        old_label = generate_button.description
        generate_button.description = "⏳ AI is coding..."

        print("=" * 60)
        print("🎨 SCREENSHOT → COMPLETE PROJECT")
        print("=" * 60)
        print(f"📷 Screenshot: {filename}\n🧩 Stack: {framework}\n📦 Project: {project_name}")
        
        try:
            print("\n🔌 Booting / Checking local Devstral Server...")
            
            # WAKE UP SERVER IF NEEDED (Fixes ConnectionRefusedError)
            if 'load_devstral_model' in globals():
                globals()['load_devstral_model']()
                
            requests.get(BASE_URL + "/models", timeout=8).raise_for_status()
            
            print("✅ Server is ready. Analyzing UI & writing code...")
            started = time.time()
            
            project = call_devstral_project(input_path, framework, instruction, project_name)
            
            print(f"⏱️ AI generation finished in {time.time() - started:.1f}s")
            
            # Write Files
            written = write_project_files(project)
            
            # Save reference screenshot inside project
            assets_dir = os.path.join(PROJECT_ROOT, "public", "assets")
            os.makedirs(assets_dir, exist_ok=True)
            shutil.copy2(input_path, os.path.join(assets_dir, f"reference-screenshot{os.path.splitext(filename)[1].lower()}"))
            
            # Zip and provide download
            zip_path = create_zip()
            
            print(f"\n✅ COMPLETE PROJECT GENERATED ({len(project_file_tree())} files)")
            for path in project_file_tree():
                print(f"  📄 {path}")
                
            display(FileLink(zip_path, result_html_prefix="\n⬇️ Download Full Project: "))

        except Exception as exc:
            print(f"\n❌ Generation failed:\n{exc}")
        finally:
            _running = False
            generate_button.disabled = False
            generate_button.description = old_label
            if os.path.exists(input_path): os.remove(input_path)

def clear_all(_):
    with output: clear_output()
    try: screenshot_upload.value = ()
    except: pass
    instruction_input.value = ""

# ------------------------------------------------------------
# 6. INITIALIZE
# ------------------------------------------------------------

generate_button.on_click(generate_project)
clear_button.on_click(clear_all)

display(header, screenshot_upload, project_name_input, framework_input, instruction_input)
display(widgets.HBox([generate_button, clear_button], layout=widgets.Layout(gap="10px")))
display(output)

HTML(value='\n    <div style="margin:20px 0 15px; padding:20px; border-radius:12px; background:linear-gradient…

FileUpload(value=(), accept='image/*', description='📷 Upload Screenshot', layout=Layout(width='260px'))

Text(value='screenshot-ui', description='Project Name:', layout=Layout(width='650px'))

Text(value='Next.js + TypeScript + Tailwind CSS', description='Stack:', layout=Layout(width='850px'))

Textarea(value='', description='Instruction:', layout=Layout(height='100px', width='850px'), placeholder='E.g.…

Output()

In [10]:
#Project Refiner & Live Preview

import os
import requests
import json
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

refine_header = widgets.HTML("<h2>✨ Project AI Refiner</h2>")
edit_instruction = widgets.Textarea(description="Instruction:", layout=widgets.Layout(width="800px"))
refine_button = widgets.Button(description="✨ Apply AI Edit", button_style="warning")
refine_output = widgets.Output()

def get_project_files():
    tree = []
    for root, _, files in os.walk(PROJECT_ROOT):
        for f in files:
            path = os.path.join(root, f)
            with open(path, 'r', encoding='utf-8', errors='ignore') as file:
                tree.append(f"File: {os.path.relpath(path, PROJECT_ROOT)}\n{file.read()[:5000]}")
    return "\n\n".join(tree)
# ============================================================
# CELL 11 - COMPLETE PROJECT AI REFINER + WORKING LIVE PREVIEW
# Multi-file editing, Live React/HTML Preview, and UI Inspector
# ============================================================

import os
import re
import json
import html
import shutil
import tempfile
import time
import uuid
import base64
import mimetypes
import zipfile
import requests
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output, FileLink

# ============================================================
# 1. LOCAL MODEL SETTINGS
# ============================================================

LOCAL_API = globals().get("DEVSTRAL_BASE_URL", "http://127.0.0.1:8000/v1")
LOCAL_MODEL = globals().get("MODEL_ID", "mistralai/Devstral-Small-2-24B-Instruct-2512")
PROJECT_ROOT = "/kaggle/working/generated_project"

os.makedirs(PROJECT_ROOT, exist_ok=True)

# ============================================================
# 2. FILE HELPERS
# ============================================================

IGNORED_DIRS = {".git", "node_modules", "__pycache__", ".next", "dist", "build", ".cache", ".preview"}

def project_files():
    result = []
    if not os.path.isdir(PROJECT_ROOT): return result
    for root, dirs, files in os.walk(PROJECT_ROOT):
        dirs[:] = [d for d in dirs if d not in IGNORED_DIRS]
        for name in files:
            path = os.path.join(root, name)
            if any(x in path.lower() for x in [".backup_", ".bak", ".tmp"]): continue
            result.append(os.path.relpath(path, PROJECT_ROOT).replace("\\", "/"))
    return sorted(result)

def read_text(path):
    with open(path, "r", encoding="utf-8") as f: return f.read()

def absolute_path(relative):
    relative = str(relative).replace("\\", "/").lstrip("/")
    parts = [p for p in relative.split("/") if p not in ("", ".", "..")]
    clean = "/".join(parts)
    root = os.path.abspath(PROJECT_ROOT)
    target = os.path.abspath(os.path.join(root, clean))
    if not target.startswith(root + os.sep): raise ValueError("Unsafe path: " + relative)
    return target

def backup(path):
    if not os.path.exists(path): return None
    stamp = time.strftime("%Y%m%d_%H%M%S")
    target = path + ".backup_" + stamp
    shutil.copy2(path, target)
    return target

def atomic_write(path, content):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=".refine_", suffix=".tmp", dir=os.path.dirname(path))
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            f.write(content)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, path)
    except Exception:
        try: os.remove(tmp)
        except: pass
        raise

# ============================================================
# 3. FRAMEWORK & ENTRY
# ============================================================

def current_framework():
    value = refine_framework_input.value.strip()
    if value: return value
    for name in ["framework_input", "FRAMEWORK", "PROJECT_FRAMEWORK"]:
        if name in globals() and hasattr(globals()[name], "value"):
            return globals()[name].value.strip()
    return ""

def framework_kind(framework):
    fw = framework.lower()
    if "next.js" in fw or "react" in fw: return "react"
    if "html" in fw or "vanilla" in fw: return "html"
    return "generic"

ENTRY_CANDIDATES = [
    "app/page.tsx", "app/page.jsx", "src/app/page.tsx", "src/app/page.jsx", 
    "pages/index.tsx", "pages/index.jsx", "src/App.tsx", "src/App.jsx", 
    "App.tsx", "App.jsx", "index.html"
]

def find_entry():
    files = project_files()
    for item in ENTRY_CANDIDATES:
        if item in files: return item
    scored = []
    for path in files:
        low = path.lower()
        name = os.path.basename(low)
        score = 0
        if name in {"page.tsx", "page.jsx", "app.tsx", "app.jsx", "index.html"}: score += 100
        if "/app/" in "/" + low: score += 30
        if low.endswith((".tsx", ".jsx", ".html")): score += 20
        if "components/" in low: score -= 20
        scored.append((score, path))
    if not scored: return None
    scored.sort(reverse=True)
    return scored[0][1]

def find_reference_screenshot():
    found = []
    for root, dirs, files in os.walk(PROJECT_ROOT):
        dirs[:] = [d for d in dirs if d not in IGNORED_DIRS]
        for name in files:
            ext = os.path.splitext(name)[1].lower()
            if ext in {".png", ".jpg", ".jpeg", ".webp"}:
                path = os.path.join(root, name)
                try: modified = os.path.getmtime(path)
                except: modified = 0
                found.append((modified, path))
    if not found: return None
    found.sort(reverse=True)
    return found[0][1]

def image_data_uri(path):
    if not path or not os.path.exists(path): return None
    mime = mimetypes.guess_type(path)[0] or "image/png"
    with open(path, "rb") as f: encoded = base64.b64encode(f.read()).decode("ascii")
    return f"data:{mime};base64,{encoded}"

# ============================================================
# 4. PROJECT CONTEXT
# ============================================================

def context_text(entry):
    ordered = []
    total = 0
    max_chars = 90000
    
    # Simple collection of all relevant code files to fit context
    for path in project_files():
        if not path.endswith(('.tsx', '.jsx', '.ts', '.js', '.html', '.css', '.json')): continue
        try:
            content = read_text(absolute_path(path))
            if total + len(content) > max_chars: continue
            ordered.append((path, content))
            total += len(content)
        except: pass

    blocks = []
    for path, content in ordered:
        blocks.append(f"==============================\nFILE: {path}\n==============================\n{content}")
    return "\n\n".join(blocks) or "(No project context.)"

# ============================================================
# 5. INSPECTOR & PREVIEW HTML
# ============================================================

def inspector_script(channel):
    code = r"""
<style>
#AI_INSPECTOR_BADGE { position:fixed; top:10px; right:10px; z-index:2147483647; padding:8px 12px; background:#111827; color:#fff; border-radius:8px; font:600 12px system-ui,sans-serif; pointer-events:none; box-shadow:0 4px 15px rgba(0,0,0,.25); }
.AI_INSPECTOR_HOVER { outline:2px dashed #2563eb !important; outline-offset:2px !important; cursor:crosshair !important; }
.AI_INSPECTOR_SELECTED { outline:3px solid #ef4444 !important; outline-offset:3px !important; }
</style>
<div id="AI_INSPECTOR_BADGE">INSPECT MODE — click an element</div>
<script>
(function(){
    var CHANNEL = __CHANNEL__;
    var selected = null;
    var hovered = null;

    function selectorOf(el){
        if(!el) return "";
        if(el.id) return "#" + el.id;
        var parts = [];
        var current = el;
        while(current && current.nodeType === 1 && current !== document.documentElement){
            var part = current.tagName.toLowerCase();
            if(current.classList && current.classList.length){
                var classes = Array.from(current.classList).filter(function(c){return !c.includes("AI_INSPECTOR");}).slice(0,2);
                if(classes.length) part += "." + classes.join(".");
            }
            var parent = current.parentElement;
            if(parent){
                var siblings = Array.from(parent.children).filter(function(node){return node.tagName === current.tagName;});
                if(siblings.length > 1) part += ":nth-of-type(" + (siblings.indexOf(current) + 1) + ")";
            }
            parts.unshift(part);
            current = parent;
            if(current === document.body){ parts.unshift("body"); break; }
        }
        return parts.join(" > ");
    }

    function detailsOf(el, selector){
        var attrs = [];
        Array.from(el.attributes || []).forEach(function(attr){
            if(attr.name !== "class" && attr.name !== "style") attrs.push(attr.name + '="' + String(attr.value).slice(0, 180) + '"');
        });
        var style = getComputedStyle(el);
        return ["TAG: " + el.tagName.toLowerCase(), "CLASS: " + (el.className || "(none)"), "TEXT: " + (el.innerText || "").slice(0,100), "ATTRIBUTES: " + attrs.join(" | "), "SELECTOR: " + selector].join("\n");
    }

    document.addEventListener("mousemove", function(event){
        var el = event.target;
        if(!el || el.id === "AI_INSPECTOR_BADGE") return;
        if(hovered && hovered !== selected) hovered.classList.remove("AI_INSPECTOR_HOVER");
        hovered = el;
        if(hovered !== selected) hovered.classList.add("AI_INSPECTOR_HOVER");
    }, true);

    document.addEventListener("click", function(event){
        var el = event.target;
        if(!el || el.id === "AI_INSPECTOR_BADGE") return;
        event.preventDefault(); event.stopPropagation(); event.stopImmediatePropagation();
        if(selected) selected.classList.remove("AI_INSPECTOR_SELECTED");
        if(hovered) hovered.classList.remove("AI_INSPECTOR_HOVER");
        selected = el;
        selected.classList.add("AI_INSPECTOR_SELECTED");
        
        window.parent.postMessage({
            type: "AI_INSPECTOR_SELECTION",
            channel: CHANNEL,
            selector: selectorOf(el),
            details: detailsOf(el, selectorOf(el))
        }, "*");
    }, true);
})();
</script>
"""
    return code.replace("__CHANNEL__", json.dumps(channel))

def build_html_preview(source, inspect, channel):
    if not inspect: return source
    script = inspector_script(channel)
    lower = source.lower()
    match = re.search(r"</body\s*>", lower)
    if match: return source[:match.start()] + script + source[match.start():]
    return source + script

def make_react_preview(entry, inspect, channel):
    modules = {}
    assets = {}
    for path in project_files():
        if path.endswith((".tsx", ".jsx", ".ts", ".js", ".json")):
            try: modules[path] = read_text(absolute_path(path))
            except: pass
            
    module_json = json.dumps(modules, ensure_ascii=False)
    asset_json = json.dumps(assets, ensure_ascii=False)
    
    global_css = ""
    for rel in ["app/globals.css", "src/app/globals.css", "src/index.css", "src/App.css", "globals.css", "index.css"]:
        try: global_css += "\n" + re.sub(r"@import\s+['\"][^'\"]+['\"]\s*;", "", re.sub(r"@tailwind\s+[^;]+;", "", read_text(absolute_path(rel))))
        except: pass

    inspector = inspector_script(channel) if inspect else ""

    document = r"""
<!doctype html>
<html>
<head>
<meta charset="utf-8">
<script src="https://cdn.tailwindcss.com"></script>
<style>html,body,#root{min-height:100%; margin:0;} __GLOBAL_CSS__</style>
</head>
<body>
<div id="root"></div>
<script src="https://unpkg.com/react@18/umd/react.development.js"></script>
<script src="https://unpkg.com/react-dom@18/umd/react-dom.development.js"></script>
<script src="https://unpkg.com/@babel/standalone/babel.min.js"></script>
<script>
(function(){
    var MODULES = __MODULES__;
    var cache = {};

    function compileModule(path){
        if(cache[path]) return cache[path].exports;
        var source = MODULES[path];
        if(!source) return {};
        var module = {exports:{}};
        cache[path] = module;

        function require(request){
            if(request === "react") return React;
            if(request === "react-dom" || request === "react-dom/client") return ReactDOM;
            if(request === "lucide-react") return new Proxy({}, {get: (t, p) => () => React.createElement("span", null, "[Icon]")});
            if(request.startsWith("next/")) return new Proxy({}, {get: (t, p) => (props) => React.createElement("div", props, props.children)});
            
            var local = null;
            if(request.startsWith("./") || request.startsWith("../")) {
                var dir = path.substring(0, path.lastIndexOf('/'));
                var resolved = dir ? dir + "/" + request.replace("./", "") : request.replace("./", "");
                var cands = [resolved, resolved+".tsx", resolved+".jsx", resolved+".ts", resolved+".js", resolved+"/index.tsx", resolved+"/index.jsx"];
                for(var c of cands) if(MODULES[c]) { local = c; break; }
            }
            if(local) return compileModule(local);
            return new Proxy({}, {get: (t, p) => (props) => React.createElement("div", props, props.children)});
        }

        try{
            var compiled = Babel.transform(source, {filename:path, presets:["env", "typescript", "react"]}).code;
            new Function("require", "module", "exports", compiled)(require, module, module.exports);
        }catch(e){
            console.error("Compile error", e);
        }
        return module.exports;
    }

    try{
        var entry = compileModule("__ENTRY__");
        var App = entry.default || entry.App || entry.Page || entry;
        ReactDOM.createRoot(document.getElementById("root")).render(React.createElement(App));
    }catch(e){
        document.getElementById("root").innerHTML = "<pre style='color:red; padding:20px;'>" + String(e) + "</pre>";
    }
})();
</script>
__INSPECTOR__
</body>
</html>
"""
    return document.replace("__MODULES__", module_json).replace("__ASSETS__", asset_json).replace("__GLOBAL_CSS__", global_css).replace("__ENTRY__", entry.replace("\\", "/")).replace("__INSPECTOR__", inspector)

def render_live_preview(inspect=False):
    framework = current_framework()
    entry = find_entry()
    if not entry:
        display(HTML("<div style='padding:14px; background:#fef3c7; color:#92400e;'>⚠️ No project entry file detected.</div>"))
        return

    channel = "inspect-" + uuid.uuid4().hex
    kind = framework_kind(framework)

    try:
        if kind == "react" and entry.lower().endswith((".tsx", ".jsx", ".ts", ".js")):
            page = make_react_preview(entry, inspect, channel)
            adapter = "React / Next.js Module Preview"
        elif entry.lower().endswith((".html", ".htm")):
            page = build_html_preview(read_text(absolute_path(entry)), inspect, channel)
            adapter = "HTML Direct Preview"
        else:
            display(HTML("<div style='padding:14px; background:#eff6ff; color:#1e3a8a;'>Live preview works for HTML/React.</div>"))
            return
    except Exception as exc:
        display(HTML(f"<div style='padding:14px; background:#fee2e2; color:#991b1b;'>❌ Preview failed:<br>{html.escape(str(exc))}</div>"))
        return

    iframe_srcdoc = html.escape(page, quote=True)
    parent_script = f"""
<script>
(function(){{
    window.addEventListener("message", function(event){{
        var data = event.data || {{}};
        if(data.type !== "AI_INSPECTOR_SELECTION" || data.channel !== "{channel}") return;
        var sel = document.querySelector(".refiner-selector-input input");
        var det = document.querySelector(".refiner-details-input textarea");
        if(sel) {{ sel.value = data.selector || ""; sel.dispatchEvent(new Event("input", {{bubbles:true}})); }}
        if(det) {{ det.value = data.details || ""; det.dispatchEvent(new Event("input", {{bubbles:true}})); }}
    }});
}})();
</script>
"""
    display(HTML(f"""
        <div style="margin-top:15px; border:2px solid #cbd5e1; border-radius:10px; overflow:hidden;">
            <div style="padding:10px 13px; background:#e2e8f0; border-bottom:1px solid #cbd5e1; font:600 13px system-ui;">
                {"🔍 Inspect Mode" if inspect else "🌐 Live Preview"}
                <span style="float:right; font-weight:400; opacity:.75;">{adapter}</span>
            </div>
            <iframe srcdoc="{iframe_srcdoc}" style="display:block; width:100%; height:720px; border:0; background:white;" sandbox="allow-scripts allow-forms allow-modals allow-same-origin"></iframe>
        </div>
        {parent_script}
    """))

# ============================================================
# 6. AI EDIT / REFINE
# ============================================================

def call_devstral_edit(framework, instruction, selector, details, entry):
    context = context_text(entry)
    tree = "\n".join(project_files()) or "(empty project)"
    
    selected = f"SELECTED ELEMENT\n================\nSelector:\n{selector}\n\nDetails:\n{details or '(none)'}" if selector else "No element selected."

    prompt = f"""You are a principal frontend engineer editing a COMPLETE MULTI-FILE PROJECT.

FRAMEWORK / STACK: {framework}
ENTRY FILE: {entry}

USER INSTRUCTION: {instruction}
{selected}

PROJECT TREE:
{tree}

RELEVANT SOURCE:
{context}

RULES:
1. Modify every file required to implement the instruction.
2. Output valid JSON containing the FULL file contents of modified files.

RETURN VALID JSON ONLY:
{{
  "summary": "description",
  "files": [
    {{ "action": "update", "path": "src/App.tsx", "content": "FULL NEW CONTENT" }}
  ]
}}
"""
    
    # Try streaming for real-time output in the notebook
    response = requests.post(
        LOCAL_API + "/chat/completions",
        json={"model": LOCAL_MODEL, "messages": [{"role": "user", "content": prompt}], "temperature": 0.08, "stream": True},
        stream=True, timeout=1200
    )
    response.raise_for_status()
    
    full_response = ""
    stream_display = display(HTML("<div style='color:gray; font-style:italic;'>AI is thinking...</div>"), display_id=True)
    
    for line in response.iter_lines():
        if line:
            decoded = line.decode('utf-8')
            if decoded.startswith('data: '):
                data_str = decoded[6:]
                if data_str == '[DONE]': break
                try:
                    chunk = json.loads(data_str)
                    if "choices" in chunk and len(chunk["choices"]) > 0:
                        content = chunk["choices"][0].get("delta", {}).get("content", "")
                        if content:
                            full_response += content
                            stream_display.update(HTML(f"<div style='background:#1e1e1e; color:#d4d4d4; padding:15px; white-space:pre-wrap; font-family:monospace; font-size:12px; max-height:400px; overflow-y:auto;'>{html.escape(full_response)}</div>"))
                except: pass

    text = full_response.strip()
    fenced = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text, re.IGNORECASE)
    if fenced: text = fenced.group(1).strip()
    first, last = text.find("{"), text.rfind("}")
    if first >= 0 and last > first: text = text[first:last + 1]

    manifest = json.loads(text)
    clean = []
    for item in manifest.get("files", []):
        action = item.get("action", "update").lower().strip()
        path = item.get("path")
        if not path or action not in {"update", "create", "delete"}: continue
        safe = "/".join(x for x in str(path).replace("\\", "/").lstrip("/").split("/") if x not in {"", ".", ".."})
        
        if action != "delete":
            clean.append({"action": action, "path": safe, "content": str(item.get("content", ""))})
        else:
            clean.append({"action": action, "path": safe})
    return clean, manifest

def apply_changes(changes):
    for item in changes:
        action = item["action"]
        path = item["path"]
        absolute = absolute_path(path)
        if action == "delete":
            os.remove(absolute)
        else:
            atomic_write(absolute, item["content"])

# ============================================================
# 7. UI WIDGETS & EVENTS
# ============================================================

refine_header = widgets.HTML("""
    <div style="margin:20px 0 14px; padding:19px; border-radius:11px; background:linear-gradient(135deg, #111827, #1d4ed8); color:white;">
        <h2 style="margin:0 0 6px;">✨ Complete Project AI Refiner</h2>
        <div style="font-size:13px; opacity:.92;">Live Edit, Inspect, and Refine your code.</div>
    </div>
""")

refine_framework_input = widgets.Text(value="Next.js + TypeScript + Tailwind", description="Stack:", layout=widgets.Layout(width="850px"))
selected_selector = widgets.Text(description="Selector:", layout=widgets.Layout(width="700px"))
selected_details = widgets.Textarea(description="Element:", layout=widgets.Layout(width="700px", height="100px"))
edit_instruction = widgets.Textarea(description="Instruction:", placeholder="Change this button to blue...", layout=widgets.Layout(width="850px", height="135px"))

refine_button = widgets.Button(description="✨ Apply AI Edit", button_style="warning", layout=widgets.Layout(width="180px", height="44px"))
preview_button = widgets.Button(description="🌐 Live Preview", button_style="info", layout=widgets.Layout(width="170px", height="44px"))
inspect_button = widgets.Button(description="🔍 Inspect & Edit", button_style="success", layout=widgets.Layout(width="185px", height="44px"))
clear_button = widgets.Button(description="✕ Clear Selection")
refine_output = widgets.Output()

selected_selector.add_class("refiner-selector-input")
selected_details.add_class("refiner-details-input")

_running = False

def on_preview(_):
    with refine_output:
        clear_output()
        print("🌐 Rendering Live Preview...")
        render_live_preview(False)

def on_inspect(_):
    with refine_output:
        clear_output()
        print("🔍 Enter Inspect Mode (Click an element in the preview)...")
        render_live_preview(True)

def clear_selection(_):
    selected_selector.value = ""
    selected_details.value = ""

def on_refine(_):
    global _running
    if _running: return
    framework, instruction, selector, details, entry = current_framework(), edit_instruction.value.strip(), selected_selector.value.strip(), selected_details.value.strip(), find_entry()
    
    with refine_output:
        clear_output()
        if not instruction: print("⚠️ Enter an edit instruction."); return
        if not entry: print("❌ Could not find project entry file."); return

        _running = True
        try:
            print("🔌 Checking server...")
            if 'load_devstral_model' in globals(): globals()['load_devstral_model']()
            requests.get(LOCAL_API + "/models", timeout=8).raise_for_status()
            
            print("🤖 Applying AI edits...")
            changes, manifest = call_devstral_edit(framework, instruction, selector, details, entry)
            apply_changes(changes)
            print("\n✅ Project updated! Refreshing preview...")
            edit_instruction.value = ""
            render_live_preview(False)
        except Exception as exc:
            print(f"\n❌ Refinement failed:\n{exc}")
        finally:
            _running = False

preview_button.on_click(on_preview)
inspect_button.on_click(on_inspect)
clear_button.on_click(clear_selection)
refine_button.on_click(on_refine)

display(refine_header, refine_framework_input)
display(widgets.HTML("<b>🎯 Selected Element</b>"), selected_selector, selected_details, clear_button)
display(widgets.HTML("<div style='font-weight:700; margin:14px 0 7px;'>✏️ Project Edit Instruction</div>"), edit_instruction)
display(widgets.HBox([refine_button, preview_button, inspect_button], layout=widgets.Layout(gap="10px")), refine_output)

HTML(value='\n    <div style="margin:20px 0 14px; padding:19px; border-radius:11px; background:linear-gradient…

Text(value='Next.js + TypeScript + Tailwind', description='Stack:', layout=Layout(width='850px'))

HTML(value='<b>🎯 Selected Element</b>')

Text(value='', description='Selector:', layout=Layout(width='700px'), _dom_classes=('refiner-selector-input',)…

Textarea(value='', description='Element:', layout=Layout(height='100px', width='700px'), _dom_classes=('refine…

Button(description='✕ Clear Selection', style=ButtonStyle())

HTML(value="<div style='font-weight:700; margin:14px 0 7px;'>✏️ Project Edit Instruction</div>")

Textarea(value='', description='Instruction:', layout=Layout(height='135px', width='850px'), placeholder='Chan…

Output()

In [16]:
# ============================================================
# KAGGLE LOCAL DEVSTRAL API PROXY
#
# Architecture:
#
#   UI
#    │
#    ▼
#  ngrok HTTPS
#    │
#    ▼
# FastAPI Proxy :8080
#    │
#    ▼
# llama-server :8000
#    │
#    ▼
# Devstral GGUF
#
# NO VPS
# NO nest_asyncio
# NO asyncio.run() in Kaggle main thread
# ============================================================

!pip install -q fastapi uvicorn httpx pyngrok

# ============================================================
# IMPORTS
# ============================================================

import asyncio
import json
import secrets
import socket
import threading
import time

import httpx
import uvicorn

from fastapi import (
    FastAPI,
    Depends,
    HTTPException,
    Request,
)

from fastapi.responses import (
    Response,
    JSONResponse,
    StreamingResponse,
)

from fastapi.security import (
    HTTPBearer,
    HTTPAuthorizationCredentials,
)

from fastapi.middleware.cors import (
    CORSMiddleware,
)

from pyngrok import ngrok


# ============================================================
# CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# NGROK TOKEN
# ------------------------------------------------------------

NGROK_AUTH_TOKEN = (
    "36w3z9ypRHu5K2lAPtwjsNNfZZE_7oyY3wRG55KBJrbZJ4oM"
)

# ------------------------------------------------------------
# LOCAL LLAMA SERVER
# ------------------------------------------------------------

LLAMA_HOST = "127.0.0.1"
LLAMA_PORT = 8000

LLAMA_BASE_URL = (
    f"http://{LLAMA_HOST}:{LLAMA_PORT}"
)

OPENAI_BASE_LOCAL = (
    f"{LLAMA_BASE_URL}/v1"
)

# ------------------------------------------------------------
# PROXY
# ------------------------------------------------------------

PROXY_HOST = "0.0.0.0"
PREFERRED_PROXY_PORT = 8080

# ------------------------------------------------------------
# UI API KEY
# ------------------------------------------------------------

PROXY_API_KEY = (
    os.getenv("PROXY_API_KEY")
    if "os" in globals()
    else None
)

if not PROXY_API_KEY:
    PROXY_API_KEY = (
        f"sk-mithu-{secrets.token_hex(16)}"
    )


# ============================================================
# PORT HELPERS
# ============================================================

def is_port_open(host, port):
    """Return True when TCP port is already listening."""

    sock = socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM,
    )

    sock.settimeout(1)

    try:
        return (
            sock.connect_ex(
                (host, port)
            ) == 0
        )

    finally:
        sock.close()


def find_free_port(start_port, end_port=8090):
    """Find a free localhost TCP port."""

    for port in range(
        start_port,
        end_port + 1
    ):

        if not is_port_open(
            "127.0.0.1",
            port
        ):

            return port

    raise RuntimeError(
        "No free proxy port found."
    )


# ============================================================
# STEP 1
# CHECK LLAMA SERVER
# ============================================================

print()
print("=" * 70)
print("🔍 CHECKING LOCAL LLAMA SERVER")
print("=" * 70)

print(
    f"Local API: {OPENAI_BASE_LOCAL}"
)

try:

    response = httpx.get(
        f"{OPENAI_BASE_LOCAL}/models",
        timeout=15.0,
    )

except Exception as e:

    raise RuntimeError(
        "\n❌ llama-server is not reachable.\n\n"
        f"Expected:\n"
        f"{OPENAI_BASE_LOCAL}/models\n\n"
        f"Error:\n{e}\n\n"
        "Start llama-server first."
    )


if response.status_code != 200:

    raise RuntimeError(
        "\n❌ llama-server returned an error.\n\n"
        f"HTTP status: {response.status_code}\n\n"
        f"Response:\n{response.text}"
    )


# ============================================================
# STEP 2
# READ MODEL LIST
# ============================================================

try:

    model_response = response.json()

except Exception as e:

    raise RuntimeError(
        "\n❌ llama-server returned invalid JSON.\n\n"
        f"Error: {e}\n\n"
        f"Raw response:\n{response.text[:5000]}"
    )


model_items = model_response.get(
    "data",
    []
)

AVAILABLE_MODELS = []

for item in model_items:

    if isinstance(item, dict):

        model_id = item.get("id")

        if model_id:

            AVAILABLE_MODELS.append(
                str(model_id)
            )


if not AVAILABLE_MODELS:

    raise RuntimeError(
        "\n❌ No model ID was returned by llama-server.\n\n"
        "Response:\n"
        + json.dumps(
            model_response,
            indent=2,
            ensure_ascii=False,
        )
    )


DEFAULT_MODEL = AVAILABLE_MODELS[0]


# ============================================================
# MODEL INFORMATION
# ============================================================

print()
print("=" * 70)
print("✅ LOCAL LLAMA SERVER DETECTED")
print("=" * 70)

print()
print("🤖 AVAILABLE MODEL(S):")

for model in AVAILABLE_MODELS:

    print(
        f"   • {model}"
    )

print()
print(
    f"🎯 DEFAULT MODEL: {DEFAULT_MODEL}"
)

print()
print(
    f"🔗 LOCAL API: {OPENAI_BASE_LOCAL}"
)


# ============================================================
# STEP 3
# FASTAPI APPLICATION
# ============================================================

app = FastAPI(
    title="Kaggle Local LLM Proxy",
    version="1.0.0",
)


# ============================================================
# CORS
# ============================================================

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)


# ============================================================
# AUTHENTICATION
# ============================================================

security = HTTPBearer(
    auto_error=True
)


def verify_api_key(
    credentials:
        HTTPAuthorizationCredentials =
        Depends(security)
):

    if (
        credentials.credentials
        != PROXY_API_KEY
    ):

        raise HTTPException(
            status_code=401,
            detail="Invalid API key",
        )

    return credentials.credentials


# ============================================================
# ROOT
# ============================================================

@app.get("/")
async def root():

    return {
        "status": "ok",
        "service": "Kaggle Local LLM Proxy",
        "architecture": (
            "ngrok -> FastAPI -> llama-server"
        ),
        "base_url": "/v1",
        "default_model": DEFAULT_MODEL,
        "models": AVAILABLE_MODELS,
    }


# ============================================================
# HEALTH
# ============================================================

@app.get("/health")
async def health():

    llama_online = False

    try:

        async with httpx.AsyncClient(
            timeout=5.0
        ) as client:

            result = await client.get(
                f"{OPENAI_BASE_LOCAL}/models"
            )

        llama_online = (
            result.status_code == 200
        )

    except Exception:
        llama_online = False

    return {
        "status": "ok",
        "proxy": "running",
        "llama_server": (
            "online"
            if llama_online
            else "offline"
        ),
        "local_api": OPENAI_BASE_LOCAL,
        "default_model": DEFAULT_MODEL,
        "models": AVAILABLE_MODELS,
    }


# ============================================================
# CONFIG
# ============================================================

@app.get("/config")
async def config():

    return {
        "base_url": "/v1",
        "default_model": DEFAULT_MODEL,
        "models": AVAILABLE_MODELS,
    }


# ============================================================
# OPENAI /v1/models
# ============================================================

@app.get("/v1/models")
async def models(
    _: str = Depends(verify_api_key)
):

    try:

        timeout = httpx.Timeout(
            connect=10.0,
            read=30.0,
            write=10.0,
            pool=10.0,
        )

        async with httpx.AsyncClient(
            timeout=timeout
        ) as client:

            upstream = await client.get(
                f"{OPENAI_BASE_LOCAL}/models"
            )

        content_type = (
            upstream.headers.get(
                "content-type",
                "application/json"
            )
        )

        return Response(
            content=upstream.content,
            status_code=upstream.status_code,
            media_type=content_type.split(";")[0],
        )

    except httpx.RequestError as e:

        raise HTTPException(
            status_code=502,
            detail=(
                "Local llama-server unavailable: "
                f"{e}"
            ),
        )


# ============================================================
# OPENAI /v1/chat/completions
# ============================================================

@app.post("/v1/chat/completions")
async def chat_completions(
    request: Request,
    _: str = Depends(verify_api_key)
):

    # --------------------------------------------------------
    # Read JSON body
    # --------------------------------------------------------

    try:

        body = await request.json()

    except Exception:

        raise HTTPException(
            status_code=400,
            detail="Invalid JSON request body",
        )


    if not isinstance(body, dict):

        raise HTTPException(
            status_code=400,
            detail="JSON body must be an object",
        )


    # --------------------------------------------------------
    # Automatic model selection
    # --------------------------------------------------------

    requested_model = body.get(
        "model"
    )

    if (
        not requested_model
        or str(requested_model).lower()
        in (
            "auto",
            "default",
        )
    ):

        body["model"] = DEFAULT_MODEL


    # --------------------------------------------------------
    # STREAM FLAG
    # --------------------------------------------------------

    stream = bool(
        body.get(
            "stream",
            False,
        )
    )


    # ========================================================
    # STREAMING
    # ========================================================

    if stream:

        async def stream_generator():

            timeout = httpx.Timeout(
                connect=20.0,
                read=None,
                write=30.0,
                pool=20.0,
            )

            try:

                async with httpx.AsyncClient(
                    timeout=timeout
                ) as client:

                    async with client.stream(
                        "POST",
                        (
                            f"{OPENAI_BASE_LOCAL}"
                            "/chat/completions"
                        ),
                        json=body,
                        headers={
                            "Content-Type":
                                "application/json",
                            "Accept":
                                "text/event-stream",
                        },
                    ) as upstream:

                        # -----------------------------
                        # Forward HTTP error
                        # -----------------------------

                        if upstream.status_code >= 400:

                            error_content = (
                                await upstream.aread()
                            )

                            yield error_content
                            return


                        # -----------------------------
                        # Forward stream
                        # -----------------------------

                        async for chunk in (
                            upstream.aiter_raw()
                        ):

                            if chunk:

                                yield chunk


            except Exception as e:

                payload = {
                    "error": {
                        "message": str(e),
                        "type": "proxy_error",
                    }
                }

                yield (
                    "data: "
                    + json.dumps(payload)
                    + "\n\n"
                ).encode(
                    "utf-8"
                )


        return StreamingResponse(
            stream_generator(),
            media_type="text/event-stream",
            headers={
                "Cache-Control":
                    "no-cache, no-transform",
                "Connection":
                    "keep-alive",
                "X-Accel-Buffering":
                    "no",
            },
        )


    # ========================================================
    # NON-STREAMING
    # ========================================================

    try:

        timeout = httpx.Timeout(
            connect=20.0,
            read=300.0,
            write=30.0,
            pool=20.0,
        )

        async with httpx.AsyncClient(
            timeout=timeout
        ) as client:

            upstream = await client.post(
                (
                    f"{OPENAI_BASE_LOCAL}"
                    "/chat/completions"
                ),
                json=body,
                headers={
                    "Content-Type":
                        "application/json",
                },
            )


        content_type = (
            upstream.headers.get(
                "content-type",
                "application/json"
            )
        )

        return Response(
            content=upstream.content,
            status_code=upstream.status_code,
            media_type=content_type.split(";")[0],
        )


    except httpx.RequestError as e:

        raise HTTPException(
            status_code=502,
            detail=(
                "Local llama-server error: "
                f"{e}"
            ),
        )


# ============================================================
# STEP 4
# SELECT PROXY PORT
# ============================================================

print()
print("=" * 70)
print("🔧 SELECTING PROXY PORT")
print("=" * 70)

if is_port_open(
    "127.0.0.1",
    PREFERRED_PROXY_PORT
):

    # Check whether existing service is our proxy
    try:

        existing = httpx.get(
            (
                f"http://127.0.0.1:"
                f"{PREFERRED_PROXY_PORT}/health"
            ),
            timeout=3,
        )

        if existing.status_code == 200:

            print(
                f"⚠️ Existing service detected on "
                f":{PREFERRED_PROXY_PORT}"
            )

            PROXY_PORT = find_free_port(
                PREFERRED_PROXY_PORT + 1
            )

            print(
                f"✅ New proxy port: "
                f"{PROXY_PORT}"
            )

        else:

            PROXY_PORT = find_free_port(
                PREFERRED_PROXY_PORT + 1
            )

    except Exception:

        PROXY_PORT = find_free_port(
            PREFERRED_PROXY_PORT + 1
        )

else:

    PROXY_PORT = PREFERRED_PROXY_PORT


# ============================================================
# STEP 5
# START NGROK
# ============================================================

print()
print("=" * 70)
print("🌍 STARTING NGROK")
print("=" * 70)

try:

    ngrok.set_auth_token(
        NGROK_AUTH_TOKEN
    )

except Exception as e:

    raise RuntimeError(
        f"❌ ngrok authentication failed:\n{e}"
    )


# Close previous tunnels
try:
    ngrok.kill()
except Exception:
    pass


try:

    tunnel = ngrok.connect(
        addr=PROXY_PORT,
        proto="http",
    )

except Exception as e:

    raise RuntimeError(
        f"❌ Failed to create ngrok tunnel:\n{e}"
    )


PUBLIC_URL = (
    tunnel.public_url
)

BASE_URL = (
    f"{PUBLIC_URL}/v1"
)


# ============================================================
# STEP 6
# START UVICORN WITHOUT uvicorn.run()
#
# This avoids the Kaggle loop_factory issue.
# ============================================================

print()
print("=" * 70)
print("🚀 STARTING FASTAPI")
print("=" * 70)

uvicorn_config = uvicorn.Config(
    app,
    host=PROXY_HOST,
    port=PROXY_PORT,
    log_level="info",
)

uvicorn_server = uvicorn.Server(
    uvicorn_config
)


def run_uvicorn():

    # Create an independent event loop
    # inside this background thread.

    loop = asyncio.new_event_loop()

    asyncio.set_event_loop(
        loop
    )

    try:

        loop.run_until_complete(
            uvicorn_server.serve()
        )

    finally:

        loop.close()


server_thread = threading.Thread(
    target=run_uvicorn,
    daemon=True,
    name="kaggle-fastapi"
)

server_thread.start()


# ============================================================
# STEP 7
# WAIT FOR FASTAPI
# ============================================================

print()
print(
    f"⏳ Waiting for FastAPI :{PROXY_PORT}..."
)

proxy_ready = False

for attempt in range(1, 21):

    try:

        health = httpx.get(
            (
                f"http://127.0.0.1:"
                f"{PROXY_PORT}/health"
            ),
            timeout=2,
        )

        if health.status_code == 200:

            proxy_ready = True
            break

    except Exception:
        pass

    time.sleep(1)


# ============================================================
# FAILED
# ============================================================

if not proxy_ready:

    raise RuntimeError(
        "\n❌ FastAPI did not start.\n\n"
        "Possible causes:\n"
        "- Port already in use\n"
        "- Uvicorn startup error\n"
        "- Application error\n"
    )


# ============================================================
# STEP 8
# TEST PUBLIC ENDPOINT
# ============================================================

print()
print("=" * 70)
print("🧪 TESTING PUBLIC PROXY")
print("=" * 70)

public_health_ok = False

try:

    public_response = httpx.get(
        f"{PUBLIC_URL}/health",
        timeout=20,
    )

    if public_response.status_code == 200:

        public_health_ok = True

        print(
            "✅ Public health endpoint:"
            f" HTTP {public_response.status_code}"
        )

    else:

        print(
            "⚠️ Public health returned:"
            f" HTTP {public_response.status_code}"
        )

except Exception as e:

    print(
        f"⚠️ Public health test failed: {e}"
    )


# ============================================================
# FINAL OUTPUT
# ============================================================

print()
print("=" * 70)

if public_health_ok:

    print(
        "🚀 KAGGLE LOCAL LLM PROXY IS READY"
    )

else:

    print(
        "⚠️ PROXY IS RUNNING, "
        "BUT PUBLIC TEST FAILED"
    )

print("=" * 70)

print()
print("🧠 ARCHITECTURE")
print("-" * 70)
print("UI")
print(" ↓")
print("ngrok HTTPS")
print(" ↓")
print(
    f"FastAPI :{PROXY_PORT}"
)
print(" ↓")
print(
    "llama-server :8000"
)
print(" ↓")
print("Devstral GGUF")

print()
print("🤖 MODEL")
print("-" * 70)

for model in AVAILABLE_MODELS:

    print(
        f"• {model}"
    )

print()
print(
    f"🎯 DEFAULT MODEL: "
    f"{DEFAULT_MODEL}"
)

print()
print("🌍 OPENAI COMPATIBLE")
print("-" * 70)
print(
    f"BASE URL: {BASE_URL}"
)

print()
print("🔑 API KEY")
print("-" * 70)
print(
    PROXY_API_KEY
)

print()
print("❤️ HEALTH")
print("-" * 70)
print(
    f"{PUBLIC_URL}/health"
)

print()
print("🤖 MODELS")
print("-" * 70)
print(
    f"{BASE_URL}/models"
)

print()
print("💬 CHAT")
print("-" * 70)
print(
    f"{BASE_URL}/chat/completions"
)

print()
print("=" * 70)
print("📌 COPY THESE INTO YOUR UI")
print("=" * 70)

print()
print(
    f"Base URL : {BASE_URL}"
)

print(
    f"API Key  : {PROXY_API_KEY}"
)

print(
    f"Model    : {DEFAULT_MODEL}"
)

print()
print("Model can also be:")
print("auto")

print()
print("=" * 70)
print("✅ llama-server is already running locally.")
print("✅ FastAPI is running in a background thread.")
print("✅ ngrok is connected.")
print("✅ Do not stop the Kaggle session.")
print("=" * 70)


🔍 CHECKING LOCAL LLAMA SERVER
Local API: http://127.0.0.1:8000/v1

✅ LOCAL LLAMA SERVER DETECTED

🤖 AVAILABLE MODEL(S):
   • mistralai/Devstral-Small-2-24B-Instruct-2512

🎯 DEFAULT MODEL: mistralai/Devstral-Small-2-24B-Instruct-2512

🔗 LOCAL API: http://127.0.0.1:8000/v1

🔧 SELECTING PROXY PORT

🌍 STARTING NGROK

🚀 STARTING FASTAPI


INFO:     Started server process [5745]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8080 (Press CTRL+C to quit)



⏳ Waiting for FastAPI :8080...
INFO:     127.0.0.1:36052 - "GET /health HTTP/1.1" 200 OK

🧪 TESTING PUBLIC PROXY
INFO:     35.252.251.134:0 - "GET /health HTTP/1.1" 200 OK
✅ Public health endpoint: HTTP 200

🚀 KAGGLE LOCAL LLM PROXY IS READY

🧠 ARCHITECTURE
----------------------------------------------------------------------
UI
 ↓
ngrok HTTPS
 ↓
FastAPI :8080
 ↓
llama-server :8000
 ↓
Devstral GGUF

🤖 MODEL
----------------------------------------------------------------------
• mistralai/Devstral-Small-2-24B-Instruct-2512

🎯 DEFAULT MODEL: mistralai/Devstral-Small-2-24B-Instruct-2512

🌍 OPENAI COMPATIBLE
----------------------------------------------------------------------
BASE URL: https://unjapanned-misha-unmeditative.ngrok-free.dev/v1

🔑 API KEY
----------------------------------------------------------------------
sk-mithu-8d6d847e2b42dfaa7d302238c53cef4a

❤️ HEALTH
----------------------------------------------------------------------
https://unjapanned-misha-unmeditative.ngro

In [11]:
# 1. OpenAI SDK ইনস্টল করা না থাকলে ইনস্টল করে নাও
!pip install openai -q

from openai import OpenAI

# তোমার জেনারেট হওয়া ক্রেডেনশিয়ালস
API_KEY = "sk-mithu-df0ce9c25c05373c56958988"
BASE_URL = "https://unjapanned-misha-unmeditative.ngrok-free.dev/v1"

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

print("🔍 Checking Proxy Connection & Models...")
try:
    # প্রথমে মডেল লিস্ট চেক করা
    models = client.models.list()
    model_id = models.data[0].id
    print(f"✅ Connection Successful! Found model: {model_id}\n")
    
    print("🤖 Sending test chat request to VPS...")
    # প্রক্সির মাধ্যমে VPS-এ চ্যাট রিকোয়েস্ট পাঠানো
    response = client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "user", "content": "Say exactly: 'Proxy is working perfectly!'"}
        ],
        max_tokens=20
    )
    
    print("✅ Response from Devstral:")
    print(f"💬 {response.choices[0].message.content}")

except Exception as e:
    print(f"❌ Test Failed. Error: {e}")

🔍 Checking Proxy Connection & Models...
❌ Test Failed. Error: The endpoint unjapanned-misha-unmeditative.ngrok-free.dev is offline.

ERR_NGROK_3200


In [12]:
#File Manager and Terminal

import os
import subprocess
import html
import ipywidgets as widgets
from IPython.display import display, clear_output, FileLink

START_DIR = "/kaggle/working"
current_dir = START_DIR

file_output = widgets.Output()
terminal_output = widgets.Output()

path_input = widgets.Text(value=START_DIR, description="Path:")
refresh_button = widgets.Button(description="🔄 Refresh", button_style="info")
command_input = widgets.Text(placeholder="ls -lah", description="$")
run_button = widgets.Button(description="▶ Run Terminal", button_style="success")

def show_files(_=None):
    global current_dir
    current_dir = path_input.value
    with file_output:
        clear_output()
        print(f"📁 {current_dir}\n" + "-"*40)
        try:
            for name in sorted(os.listdir(current_dir)):
                full_path = os.path.join(current_dir, name)
                if os.path.isdir(full_path): print(f"📁 {name}/")
                else: display(FileLink(full_path, result_html_prefix="📄 "))
        except Exception as e:
            print(f"Error: {e}")

def run_terminal(_):
    cmd = command_input.value
    with terminal_output:
        print(f"\n$ {cmd}")
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=current_dir)
        print(result.stdout if result.stdout else result.stderr)

refresh_button.on_click(show_files)
run_button.on_click(run_terminal)

display(widgets.HTML("<h3>🗂️ File Manager</h3>"), widgets.HBox([path_input, refresh_button]), file_output)
display(widgets.HTML("<h3>💻 Terminal</h3>"), widgets.HBox([command_input, run_button]), terminal_output)
show_files()

HTML(value='<h3>🗂️ File Manager</h3>')

Output()

HTML(value='<h3>💻 Terminal</h3>')

Output()

In [13]:
#Quick Zip Utility

import os
import shutil
from IPython.display import display, FileLink

FOLDER_TO_ZIP = "/kaggle/working/generated_project"
zip_base = "/kaggle/working/generated_project"

if os.path.isdir(FOLDER_TO_ZIP):
    print(f"📦 Zipping {FOLDER_TO_ZIP}...")
    zip_path = shutil.make_archive(zip_base, "zip", root_dir=FOLDER_TO_ZIP)
    display(FileLink(zip_path, result_html_prefix="⬇️ Download ZIP: "))
else:
    print("❌ Nothing to zip yet! Generate a project first.")

📦 Zipping /kaggle/working/generated_project...


/kaggle/working/generated_project.zip